In [3]:
import pandas as pd
import numpy as np
from collections import Counter
from catboost import CatBoostRegressor

train=pd.read_csv('train_data.csv').drop(['ID'],axis =1)
test = pd.read_csv('test_data.csv')


counter = Counter()
counter.update(train['City A'].tolist())
city_toID = {}
weather = {'Fog':0,'Clear':1,'Rain':2,'Snow':3}
k = 0
for i,_ in counter.items():
    city_toID[i] = k
    k += 1
train['City A'] = train['City A'].map(city_toID).astype('category')
test['City A'] = test['City A'].map(city_toID).astype('category')
train['City B'] = train['City B'].map(city_toID).astype('category')
test['City B'] = test['City B'].map(city_toID).astype('category')
train['Weather'] = train['Weather'].map(weather).astype('category')
test['Weather'] = test['Weather'].map(weather).astype('category')

In [4]:
cat = CatBoostRegressor(iterations = 1500,learning_rate = 0.008,max_depth = 6, rsm=0.9,cat_features=['City A','City B','Weather'])
cat.fit(train.drop(['deliver_time'],axis = 1),train['deliver_time'])

0:	learn: 166.0159172	total: 58.6ms	remaining: 1m 27s
1:	learn: 164.7609214	total: 70.2ms	remaining: 52.6s
2:	learn: 163.5537667	total: 79.2ms	remaining: 39.5s
3:	learn: 162.3846893	total: 87.5ms	remaining: 32.7s
4:	learn: 161.1935278	total: 94.4ms	remaining: 28.2s
5:	learn: 160.0107610	total: 99.6ms	remaining: 24.8s
6:	learn: 158.8800164	total: 103ms	remaining: 22.1s
7:	learn: 157.7188775	total: 108ms	remaining: 20.2s
8:	learn: 156.6246813	total: 114ms	remaining: 18.9s
9:	learn: 155.4986121	total: 119ms	remaining: 17.8s
10:	learn: 154.3627607	total: 123ms	remaining: 16.7s
11:	learn: 153.2353080	total: 132ms	remaining: 16.3s
12:	learn: 152.1233034	total: 136ms	remaining: 15.6s
13:	learn: 151.0426139	total: 140ms	remaining: 14.8s
14:	learn: 149.9096239	total: 147ms	remaining: 14.5s
15:	learn: 148.8402771	total: 151ms	remaining: 14s
16:	learn: 147.7595667	total: 154ms	remaining: 13.5s
17:	learn: 146.6883964	total: 158ms	remaining: 13s
18:	learn: 145.6257584	total: 163ms	remaining: 12.7s


CatBoostRegressor(cat_features=['City A', 'City B', 'Weather'], iterations=1500, learning_rate=0.008, loss_function='RMSE', max_depth=6, rsm=0.9)

In [5]:
s = test[test['Weather'] == 0]

t2 = pd.DataFrame({
    'subtaskID':2,
    'datapointID':test['ID'],
    'answer':cat.predict(test.drop('ID',axis=1))
})
t1 = pd.DataFrame({
    'subtaskID':[1],
    'datapointID':[1],
    'answer': [int(len(s[s['City A'] == city_toID['Barlad']]))]
})
pd.concat([t1,t2]).to_csv('submit.csv')